# 🧠 Micrograd from Scratch

### Author: Rahul Kurup

This notebook implements a minimal **automatic differentiation engine** and builds a **neural network (MLP)** from scratch.

---

## 🚀 Features
- Scalar-based autograd engine
- Backpropagation using topological sort
- Activation functions: tanh, ReLU, exp
- Neural network: Neuron → Layer → MLP
- Visualization support (Graphviz)

---

## 📦 1. Core Autograd Engine

In [ ]:
import math

class Value:
    def __init__(self, data, children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __pow__(self, other):
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += other * (self.data**(other-1)) * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        t = (math.exp(2*self.data)-1)/(math.exp(2*self.data)+1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()

        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)

        build(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


## 🧪 2. Test Autograd Engine

In [ ]:
a = Value(2.0)
b = Value(3.0)

c = a * b + a
c.backward()

print("c:", c)
print("grad a:", a.grad)
print("grad b:", b.grad)

## 🧠 3. Neural Network Components

In [ ]:
import random

class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        act = sum((wi*xi for wi,xi in zip(self.w,x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs)==1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0


## 🔥 4. Training Example

In [ ]:
model = MLP(3, [4,4,1])

data = [
    ([2.0,3.0,-1.0], 1.0),
    ([3.0,-1.0,0.5], -1.0),
    ([0.5,1.0,1.0], -1.0),
    ([1.0,1.0,-1.0], 1.0)
]

for epoch in range(20):
    total_loss = 0

    for x, y in data:
        pred = model(x)
        loss = (pred - y)**2

        model.zero_grad()
        loss.backward()

        for p in model.parameters():
            p.data += -0.05 * p.grad

        total_loss += loss.data

    print(f"Epoch {epoch}: Loss {total_loss}")

## 🎯 Final Output

In [ ]:
for x, y in data:
    print(x, model(x).data)